In [11]:
from datetime import datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import kneighbors_graph
from scipy.spatial.distance import pdist, squareform
from scipy.sparse.csgraph import connected_components, shortest_path
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors

# Phase 5: Lifting paths back to macroeconomic variables

### 8.3 Linear lift for interpolated latent paths

In [ ]:
# A line in diffusion coordinate space gamma

# mapped to macro space by finding m closes points to line and making lifted weight a linear combination

# Need to split the line into discrete points, break into chunks of length approximately the average distance between points in diffusion space
# for each point find m nearest neighbours - how to do this without computing whole knn graph?


def diffusion_map(data, eps, alpha=1, k=4):
    Dsq = squareform(pdist(data)**2)
    Wm = np.exp(-Dsq/eps); q = Wm.sum(1)
    Wa = Wm/np.outer(q**alpha, q**alpha)
    da = Wa.sum(1); Dis = 1.0/np.sqrt(da)
    S = Dis[:, None]*Wa*Dis[None, :]                
    w, v = np.linalg.eigh(S)
    idx = np.argsort(w)[::-1]; w, v = w[idx], v[:, idx]
    phi = Dis[:, None]*v                             
    Psi = phi[:, 1:k+1]*w[1:k+1]                     
    return {"evals": w, "Psi": Psi, "phi": phi, "W": Wm, "degrees": q}


df = pd.read_parquet("./datasets/joint_df_quantile.parquet")
# df = df.clip(-3, 3); 
Z = df.to_numpy()[:, :-1]; N = Z.shape[0]
dates = df.index; variables = df.columns
diff = diffusion_map(Z, eps=3, k=3); Psi = diff["Psi"]


In [15]:
endpoint_pairs = {"GFC": ("2006-03-01", "2008-10-01"), "COVID": ("2019-07-01", "2020-04-01"), 
                  "Fiscal Tightening": ("2019-04-01", "2022-04-01"), "Recession Trough": ("1977-01-01", "1982-07-01")}

In [101]:
def local_neighbourhood_lifting(Z, Psi, start_idx, end_idx, m, tau):
    D_Psi = squareform(pdist(Psi)); median_D = np.median(D_Psi[D_Psi > 0])
    start_point, end_point = Psi[start_idx, :], Psi[end_idx, :]
    hops = max(3, int(np.linalg.norm(end_point-start_point) // (2*median_D)))
    gamma = np.linspace(start_point, end_point, num=hops)
    nn = NearestNeighbors(n_neighbors=m); nn.fit(Psi)
    distances, indices = nn.kneighbors(gamma)
    a = np.exp(-distances**2/tau) / np.sum(np.exp(-distances**2 / tau), axis=1, keepdims=True)
    points = Z[indices, :]; z_hat = np.sum(a[:, :, None] * points, axis=1)
    z_hat[0], z_hat[-1] = Z[start_idx], Z[end_idx]
    return z_hat

In [ ]:
z_hat = local_neighbourhood_lifting(Z, Psi, 20, 25, m=10, tau=3)

array([[0.78301887, 0.96361186, 0.95013477, 0.39892183, 0.96495957,
        0.94474394, 0.76145553, 0.09568733, 0.10107817, 0.06738544,
        0.9703504 , 0.98382749, 0.95687332, 0.87466307, 0.9690027 ,
        0.89218329, 0.95283019, 0.91105121, 0.87061995, 0.53571429,
        0.42318059, 0.80660377, 0.63814016, 0.83625337, 0.30256065,
        0.62331536, 0.65768194, 0.15700809, 0.09299191, 0.4245283 ,
        0.35579515, 0.34097035, 0.28975741, 0.23045822, 0.11994609,
        0.33692722, 0.33018868, 0.33692722, 0.1819407 , 0.75067385,
        0.45619946, 0.26415094, 0.27088949, 0.10916442, 0.56940701],
       [0.50739453, 0.92210598, 0.90795872, 0.61712945, 0.90001239,
        0.89003785, 0.80391048, 0.31795727, 0.38315363, 0.19579299,
        0.92480153, 0.92480937, 0.90404504, 0.49455543, 0.70192651,
        0.71143639, 0.66268686, 0.68548901, 0.5723807 , 0.53571429,
        0.76939528, 0.85176232, 0.82299207, 0.67481741, 0.28832643,
        0.64987768, 0.57595416, 0.21649856, 0.1

In [109]:
def baker_lift(Z, Psi):
    N = Psi.shape[0]; C = np.linalg.lstsq(Psi, Z, rcond=None)[0]
    error_norm = np.linalg.norm(Z - Psi@C, axis=1)**2
    RMSE = np.sqrt(np.sum(error_norm)/N)
    return C.T, RMSE


In [110]:
baker_lift(Z, Psi)[1]

3.659210601340778